<a href="https://colab.research.google.com/github/Mukundan-T/seqADAGE/blob/master/Py/muk_transfer_learning/enrichment/Mukundan_pathway_enrichment_kegg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Evaluation - Pathway Enrichment Analysis

### Mukundan Thanigaivelan

### June 12, 2026

This notebook performs pathway enrichment analysis for the E. coli and S. aureus genes.

## 0. Connect to GitHub

In [1]:
!git clone https://github.com/Mukundan-T/seqADAGE.git

Cloning into 'seqADAGE'...
remote: Enumerating objects: 778, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 778 (delta 164), reused 60 (delta 46), pack-reused 526 (from 2)
Receiving objects: 100% (778/778), 56.75 MiB | 14.03 MiB/s, done.
Resolving deltas: 100% (414/414), done.
Updating files: 100% (161/161), done.


In [2]:
%cd seqADAGE/Py/muk_transfer_learning

/content/seqADAGE/Py/muk_transfer_learning


In [3]:
!pip install -qq biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.1 MB/s eta 0:00:00


## 1. Loading classes & modules

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [10]:
# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
import time
import tensorflow as tf
from Bio import SeqIO
import requests

In [7]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  0
True


In [8]:
ec_prot_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/ec_proteins.faa'
sa_prot_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/sa_proteins.faa'

## S. aureus

### Extract only genes used in model in FASTA

In [ ]:
# Load S. aureus filtered compendium
sa_df = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/sa_usa300_lcn01_medmean_normal_muk.csv',
  index_col = 0
)
sa_df.shape

(2174, 2181)

In [ ]:
def make_filtered_fasta(genes, input_file, output_file):
  """
  Take a set of genes to keep, an input FASTA file, and an output
  file to save the FASTA for the genes that appear in the given set.
  """
  with open(output_file, "w") as out:
    for record in SeqIO.parse(input_file, "fasta"):
      if record.id in genes:
        SeqIO.write(record, out, "fasta")

make_filtered_fasta(
  set(sa_df.index.astype(str)),
  sa_prot_fasta,
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/sa_filtered.faa'
)

In [ ]:
# Number of genes in new FASTA - matches correctly
sum(1 for _ in SeqIO.parse('/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/sa_filtered.faa', "fasta"))

2174

### Process KO Numbers

In [44]:
# Read in KO numbers
ko = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/sa_ko.txt',
  sep = '\t',
  names = ['Gene', 'KO']
)
ko.head()

,Gene,KO
0,12b493b74dec297122c11824ed05ba01_3,K02313
1,12b493b74dec297122c11824ed05ba01_5,K02338
2,12b493b74dec297122c11824ed05ba01_7,K14761
3,12b493b74dec297122c11824ed05ba01_9,K03629
4,12b493b74dec297122c11824ed05ba01_11,K02470


In [45]:
# Get KO --> Pathway mapping
response = requests.get('https://rest.kegg.jp/link/pathway/ko')
response.raise_for_status()

In [46]:
# Convert mapping into dataframe
mapping = []

for line in response.text.strip().split('\n'):
  ko_id, pathway = line.split('\t')
  mapping.append(
    {
      'KO': ko_id.replace('ko:', ''),
      'Pathway': pathway.replace('path:', '')
    }
  )

mapping = pd.DataFrame(mapping)
mapping.head()

,KO,Pathway
0,K00001,map00010
1,K00001,ko00010
2,K00002,map00010
3,K00002,ko00010
4,K00016,map00010


In [47]:
# Merge mapping into KO numbers dataframe
gene_pathways = ko.merge(mapping, on = 'KO')
gene_pathways.head()

,Gene,KO,Pathway
0,12b493b74dec297122c11824ed05ba01_3,K02313,map02020
1,12b493b74dec297122c11824ed05ba01_3,K02313,ko02020
2,12b493b74dec297122c11824ed05ba01_3,K02313,map04112
3,12b493b74dec297122c11824ed05ba01_3,K02313,ko04112
4,12b493b74dec297122c11824ed05ba01_5,K02338,map03030


In [48]:
# Get pathway names
response = requests.get('https://rest.kegg.jp/list/pathway')
response.raise_for_status()

In [49]:
names = []

for line in response.text.strip().split('\n'):
  pid, name = line.split('\t')
  names.append(
    {
      'Pathway': pid.replace('path:', ''),
      'PathwayName': name
    }
  )

names = pd.DataFrame(names)
names.head()

,Pathway,PathwayName
0,map01100,Metabolic pathways
1,map01110,Biosynthesis of secondary metabolites
2,map01120,Microbial metabolism in diverse environments
3,map01200,Carbon metabolism
4,map01210,2-Oxocarboxylic acid metabolism


In [50]:
# Merge pathway names into dataframe
gene_pathways = gene_pathways.merge(names, on = 'Pathway')
gene_pathways.head()

,Gene,KO,Pathway,PathwayName
0,12b493b74dec297122c11824ed05ba01_3,K02313,map02020,Two-component system
1,12b493b74dec297122c11824ed05ba01_3,K02313,map04112,Cell cycle - Caulobacter
2,12b493b74dec297122c11824ed05ba01_5,K02338,map03030,DNA replication
3,12b493b74dec297122c11824ed05ba01_5,K02338,map03430,Mismatch repair
4,12b493b74dec297122c11824ed05ba01_5,K02338,map03440,Homologous recombination


In [51]:
# Format the information into the dataframe desired
def join_genes(genes):
  """
  Return semicolon-delimited gene identifiers.
  """
  return ';'.join(sorted(set(genes)))

summary = (
    gene_pathways
    .groupby(['Pathway', 'PathwayName'])
    .agg(
        Length = ('Gene', 'count'),
        Genes = ('Gene', join_genes)
    )
    .reset_index()
)

# Combine two columns into one
summary['KEGGPathID'] = summary['Pathway'] + ":" + summary['PathwayName']

# Keep only these three columns
summary = summary[['KEGGPathID', 'Length', 'Genes']]

summary.head()

,KEGGPathID,Length,Genes
0,map00010:Glycolysis / Gluconeogenesis,35,12b493b74dec297122c11824ed05ba01_1254;12b493b7...
1,map00020:Citrate cycle (TCA cycle),21,12b493b74dec297122c11824ed05ba01_2093;12b493b7...
2,map00030:Pentose phosphate pathway,20,12b493b74dec297122c11824ed05ba01_1000;12b493b7...
3,map00040:Pentose and glucuronate interconversions,8,12b493b74dec297122c11824ed05ba01_2362;12b493b7...
4,map00051:Fructose and mannose metabolism,13,12b493b74dec297122c11824ed05ba01_1446;12b493b7...


In [53]:
# Save to text file
summary.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/sa_kegg_pathways.txt',
  sep = '\t',
  index = False
)

## E. coli

### Extract only genes used in model in FASTA

In [ ]:
# Load E. coli filtered compendium
ec_df = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ecmg_lcn01_medmean_normal_muk.csv',
  index_col = 0
)
ec_df.shape

(3608, 10233)

In [ ]:
def make_filtered_fasta(genes, input_file, output_file):
  """
  Take a set of genes to keep, an input FASTA file, and an output
  file to save the FASTA for the genes that appear in the given set.
  """
  with open(output_file, "w") as out:
    for record in SeqIO.parse(input_file, "fasta"):
      if record.id in genes:
        SeqIO.write(record, out, "fasta")

make_filtered_fasta(
  set(ec_df.index.astype(str)),
  ec_prot_fasta,
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_filtered.faa'
)

In [ ]:
# Number of genes in new FASTA - matches correctly
sum(1 for _ in SeqIO.parse('/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_filtered.faa', "fasta"))

3608